# Federated Governance — Dev Log

## Objetivo

Agrega métricas de governança entre múltiplos "nós" organizacionais SEM
centralizar o dado bruto: cada nó só expõe um `NodeReport` já agregado —
nenhum `TrustScoreResult` individual cruza a fronteira do nó.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.federated_governance.federation import aggregate_federation, build_node_report
from core.policy_engine.engine import evaluate
from core.trust_score.scorer import compute_trust_score
from shared.schemas import DataCategory, LegalBasis, PIIDetectionResult

neutral_pii = PIIDetectionResult(findings=[], has_sensitive_data=False, summary="sem PII")

# Nó A: filial com decisões majoritariamente de baixo risco.
scores_a = []
for _ in range(5):
    decisions = evaluate(data_categories=[DataCategory.PERSONAL], legal_basis=LegalBasis.LEGITIMATE_INTEREST)
    scores_a.append(compute_trust_score(pii_result=neutral_pii, policy_decisions=decisions))

# Nó B: filial com um caso de alto risco (biometria sem revisão).
decisions_b = evaluate(
    data_categories=[DataCategory.SENSITIVE], legal_basis=LegalBasis.NOT_DETERMINED,
    context={"data_subtype": "biometric", "automated_decision": True, "human_review": False},
)
scores_b = [compute_trust_score(pii_result=neutral_pii, policy_decisions=decisions_b)]

node_a = build_node_report("Filial SP", scores_a)
node_b = build_node_report("Filial RJ", scores_b)
federation = aggregate_federation([node_a, node_b])

print(f"Nó '{node_a.node_id}': {node_a.total_evaluations} avaliações, score médio {node_a.avg_trust_score}")
print(f"Nó '{node_b.node_id}': {node_b.total_evaluations} avaliações, score médio {node_b.avg_trust_score}")
print()
print(federation.summary)

Nó 'Filial SP': 5 avaliações, score médio 100.0
Nó 'Filial RJ': 1 avaliações, score médio 5.0

Federação de 2 nó(s), 6 avaliação(ões) no total. Trust score médio ponderado: 84.2/100. 1 decisão(ões) DENY no total. Nó com pior score médio: 'Filial RJ' (5.0); melhor: 'Filial SP' (100.0).


A `Filial RJ` tem só 1 avaliação (de alto risco), mas isso não domina a
média ponderada global — o peso é proporcional ao volume de cada nó,
exatamente o comportamento esperado de uma agregação federada honesta.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/federated_governance/tests -v
```

8/8 testes passando, incluindo `NodeReport` construído a partir de
`TrustScoreResult` real (composição completa `policy_engine` ->
`trust_score` -> `federated_governance`).

## Handoff Summary da Onda 4 (fecha o V2 completo)

Com os 7 módulos desta onda, as **19 capacidades do V2** (as 20 linhas do
ROADMAP original, com GraphRAG/Regulatory Knowledge Graph consolidados)
estão implementadas com testes reais — mesmo padrão de rigor do V1: sem
mock nos motores de produção, sem lógica de domínio reimplementada entre
módulos, limitações documentadas honestamente em vez de escondidas.